# Notebook 02: Moments and Circuit Assembly

Cirq structures circuits into discrete time-slices called Moments, representing operations that execute concurrently.

---

## Learning Objectives
1. Understand the `cirq.Moment` data structure.
2. Build circuits from collections of moments.
3. Compare insertion strategies (`EARLIEST` vs `NEW`).
4. Inspect circuit depth and moment sequencing.


---
## Real-World Applications & Modern Use Cases

Moment-based circuit synchronization is leveraged in commercial quantum computing for:
- **Decoherence Minimization:** Grouping concurrent non-interacting gates into strict time slices, ensuring qubits spend minimal time idling subject to $T_1$ relaxation and $T_2$ dephasing.
- **Dynamical Decoupling Sequences:** Systematically inserting periodic refocusing pulses into idle moments during multi-qubit algorithm execution to suppress low-frequency environmental magnetic noise.


In [1]:
import cirq

q0, q1, q2 = cirq.LineQubit.range(3)
print("Line qubits initialized: q(0), q(1), q(2)")


Line qubits initialized: q(0), q(1), q(2)


---
## Section 1: Explicit Moment Construction


In [2]:
# Moment 1: Concurrent Hadamards on q0 and q1
moment_1 = cirq.Moment(cirq.H(q0), cirq.H(q1))

# Moment 2: Two-qubit entangling gate
moment_2 = cirq.Moment(cirq.CNOT(q0, q1))

# Moment 3: Single-qubit operation on q2
moment_3 = cirq.Moment(cirq.X(q2))

circuit = cirq.Circuit(moment_1, moment_2, moment_3)
print("Explicit Moments Circuit:")
print(circuit)


Explicit Moments Circuit:
0: ───H───@───────
          │
1: ───H───X───────

2: ───────────X───


---
## Section 2: Iterating Over Moments


In [3]:
print(f"Total Moments (Circuit Depth): {len(circuit)}")
for i, moment in enumerate(circuit):
    print(f"Moment {i}: {moment}")


Total Moments (Circuit Depth): 3
Moment 0:   ╷ 0 1
╶─┼─────
0 │ H H
  │
Moment 1:   ╷ 0 1
╶─┼─────
0 │ @─X
  │
Moment 2:   ╷ 2
╶─┼───
0 │ X
  │


---
## Section 3: Automatic Moment Scheduling via Insertion Strategies


In [4]:
scheduled_circuit = cirq.Circuit()

# Insert operations using default EARLIEST strategy
scheduled_circuit.append([cirq.H(q0), cirq.H(q1)], strategy=cirq.InsertStrategy.EARLIEST)
scheduled_circuit.append(cirq.X(q2), strategy=cirq.InsertStrategy.EARLIEST)
scheduled_circuit.append(cirq.CNOT(q0, q1), strategy=cirq.InsertStrategy.EARLIEST)

print("Automatically Scheduled Circuit:")
print(scheduled_circuit)
print(f"Resulting Depth: {len(scheduled_circuit)}")


Automatically Scheduled Circuit:
0: ───H───@───
          │
1: ───H───X───

2: ───X───────
Resulting Depth: 2
